# AI Avatar Studio — MiniMax H3 Colab GPU Worker (L4 / G4 版)

此 Notebook 整合自 [dseditor/h3-colab-skill](https://github.com/dseditor/h3-colab-skill)，透過 **ComfyUI + MiniMax H3** 提供電影級人物動態與聲音克隆。

> **硬體需求**：需 **Google Colab Pro / AI Pro**，並將執行階段切換為 **L4 GPU（或 G4 GPU）**（免費 T4 顯存不足以執行 MiniMax H3）。

In [ ]:
#@title 1. 檢查 GPU、拉取 h3-colab-skill 與安裝 ComfyUI
!nvidia-smi
!pip install -q fastapi uvicorn python-multipart httpx websockets yt-dlp f5-tts openai-whisper
!apt-get -qq update && apt-get -qq install -y ffmpeg git-lfs
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb

import os
if not os.path.exists("/content/h3-colab-skill"):
    !git clone -q https://github.com/dseditor/h3-colab-skill.git /content/h3-colab-skill
if not os.path.exists("/content/ComfyUI"):
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    !pip install -q -r /content/ComfyUI/requirements.txt

print("✅ 基礎環境與 h3-colab-skill 準備完成")

In [ ]:
#@title 2. 啟動 ComfyUI 並使用 h3-colab-skill 安裝 MiniMax H3 節點與模型 (首次約 5~8 分鐘)
import subprocess, sys, time, httpx
sys.path.insert(0, "/content/h3-colab-skill/h3-multimodal-colab/scripts")

# 先背景啟動 ComfyUI 讓 h3local.py setup 可以連線檢查與安裝節點
subprocess.run(["pkill", "-f", "/content/ComfyUI/main.py"], check=False)
subprocess.Popen(
    ["python", "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188"],
    stdout=open("/content/comfyui.log", "w"),
    stderr=subprocess.STDOUT,
)
for _ in range(30):
    try:
        if httpx.get("http://127.0.0.1:8188/system_stats", timeout=2).status_code == 200:
            break
    except Exception:
        time.sleep(2)

!python /content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py --comfy http://127.0.0.1:8188 --comfy-dir /content/ComfyUI --models-dir /content/ComfyUI/models setup
print("✅ MiniMax H3 模型與 ComfyUI 節點設定完成")

In [ ]:
#@title 3. 啟動 ComfyUI + FastAPI Bridge + Cloudflare Tunnel
import json
import re
import subprocess
import sys
import threading
import time
import uuid
from pathlib import Path
import httpx
import torch
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

sys.path.insert(0, "/content/h3-colab-skill/h3-multimodal-colab/scripts")

API_TOKEN = ""  #@param {type:"string"}
TAILSCALE_AUTHKEY = ""  #@param {type:"string"}
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
NGROK_DOMAIN = ""  #@param {type:"string"}
CLOUDFLARE_TUNNEL_TOKEN = ""  #@param {type:"string"}
WORK_DIR = Path("/content/h3_worker_tmp")
WORK_DIR.mkdir(parents=True, exist_ok=True)
COMFY_URL = "http://127.0.0.1:8188"

# 重啟 ComfyUI 確保新安裝的 Custom Nodes 已載入
subprocess.run(["pkill", "-f", "/content/ComfyUI/main.py"], check=False)
time.sleep(2)
subprocess.Popen(
    ["python", "/content/ComfyUI/main.py", "--listen", "127.0.0.1", "--port", "8188"],
    stdout=open("/content/comfyui.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(8)

app = FastAPI(title="Colab MiniMax H3 Worker")
H3_JOBS = {}
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_token(authorization: str | None):
    if API_TOKEN.strip() and authorization != f"Bearer {API_TOKEN.strip()}":
        raise HTTPException(status_code=401, detail="Invalid API token")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    verify_token(authorization)
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"status": "ok", "gpu": gpu_name, "engines": ["MiniMax-H3-TTS", "MiniMax-H3-Video"]}

def _run_f5_tts_job(job_id: str, job_dir: Path, ref_path: Path, gen_text: str, speed: float, ref_text: str):
    try:
        import whisper
        resolved_ref = ref_text.strip()
        if not resolved_ref:
            wm = whisper.load_model("base")
            resolved_ref = (wm.transcribe(str(ref_path), fp16=torch.cuda.is_available()).get("text") or "").strip()
        if resolved_ref and resolved_ref[-1] not in "。！？.!?,，":
            resolved_ref += "."
        cmd = [
            "f5-tts_infer-cli", "--model", "F5TTS_v1_Base",
            "--ref_audio", str(ref_path), "--gen_text", gen_text,
            "--speed", str(speed), "--nfe_step", "32",
            "--cross_fade_duration", "0.15", "--remove_silence",
            "--output_dir", str(job_dir), "--output_file", "f5_out.wav",
        ]
        if resolved_ref:
            cmd.extend(["--ref_text", resolved_ref])
        proc = subprocess.run(cmd, capture_output=True, text=True)
        out_wav = job_dir / "f5_out.wav"
        if proc.returncode != 0 or not out_wav.exists():
            H3_JOBS[job_id] = {"status": "failed", "error": proc.stderr[-500:]}
        else:
            H3_JOBS[job_id] = {"status": "done", "file": str(out_wav)}
    except Exception as exc:
        H3_JOBS[job_id] = {"status": "failed", "error": str(exc)}

def _run_h3_tts_job(job_id: str, job_dir: Path, manifest_path: Path):
    try:
        proc = subprocess.run(
            ["python", "/content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py", "--comfy", COMFY_URL, "--comfy-dir", "/content/ComfyUI", "--models-dir", "/content/ComfyUI/models", "run", str(manifest_path), "--confirmed"],
            capture_output=True,
            text=True,
        )
        wavs = list((job_dir / "out").glob("*.wav")) if (job_dir / "out").exists() else []
        if proc.returncode != 0 or not wavs:
            H3_JOBS[job_id] = {"status": "failed", "error": (proc.stdout + proc.stderr)[-600:]}
        else:
            H3_JOBS[job_id] = {"status": "done", "file": str(wavs[0])}
    except Exception as exc:
        H3_JOBS[job_id] = {"status": "failed", "error": str(exc)}

@app.post("/api/tts/start")
async def h3_tts_start(
    text: str = Form(...),
    language: str = Form("zh"),
    speed: float = Form(1.0),
    ref_text: str = Form(""),
    ref_audio: UploadFile = File(...),
    engine: str = Form("minimax_h3"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    job_dir = WORK_DIR / f"tts_{req_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    ref_path = job_dir / "ref.wav"
    raw_ref = job_dir / f"raw_{ref_audio.filename}"
    raw_ref.write_bytes(await ref_audio.read())
    trim_sec = "4.5" if engine == "hybrid" else "6"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(raw_ref),
        "-af", "silenceremove=start_periods=1:start_silence=0.05:start_threshold=-40dB,areverse,silenceremove=start_periods=1:start_silence=0.05:start_threshold=-40dB,areverse",
        "-t", trim_sec, "-ar", "24000", "-ac", "1", str(ref_path)
    ], check=True)

    clean_text = re.sub(r"\s+", " ", text.replace("\n", "，")).strip()
    if clean_text and clean_text[-1] not in "。！？.!?,，":
        clean_text += "。"

    if engine == "hybrid":
        H3_JOBS[req_id] = {"status": "running"}
        threading.Thread(
            target=_run_f5_tts_job,
            args=(req_id, job_dir, ref_path, clean_text, speed, ref_text),
            daemon=True,
        ).start()
        return {"job_id": req_id, "status": "running"}

    lang_tag = "English" if language == "en" else "Taiwanese Mandarin"
    tts_prompt = (
        "subject_definitions:\n"
        "<Subject 1> is an adult speaker with clear, natural articulation.\n"
        "<Audio 1> is the voice-timbre reference for <Subject 1> (S1).\n\n"
        "summary:\n"
        "[reference generation + audio reference] <Subject 1> delivers a clear spoken line.\n\n"
        "retention_analysis:\n"
        "<Audio 1>: reference - its vocal timbre guides the dialogue delivery of <Subject 1>.\n\n"
        "detailed_description:\n"
        f"[Shot 1] <Subject 1> (S1) speaks clearly and steadily, <d>[{lang_tag}] {clean_text}</d>"
    )

    manifest_path = job_dir / "manifest.json"
    manifest = {
        "jobs": [{
            "id": req_id,
            "type": "tts",
            "prompt": tts_prompt,
            "voices": ["ref.wav"],
            "lead_in": True,
        }]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    H3_JOBS[req_id] = {"status": "running"}
    threading.Thread(target=_run_h3_tts_job, args=(req_id, job_dir, manifest_path), daemon=True).start()
    return {"job_id": req_id, "status": "running"}

@app.get("/api/tts/status/{job_id}")
def h3_tts_status(job_id: str, authorization: str | None = Header(default=None)):
    verify_token(authorization)
    job = H3_JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return {"job_id": job_id, "status": job["status"], "error": job.get("error")}

@app.get("/api/tts/result/{job_id}")
def h3_tts_result(job_id: str, authorization: str | None = Header(default=None)):
    verify_token(authorization)
    job = H3_JOBS.get(job_id)
    if not job or job.get("status") != "done":
        raise HTTPException(status_code=404, detail="Result not ready")
    return FileResponse(job["file"], media_type="audio/wav", filename="h3_tts.wav")

@app.post("/api/tts")
async def h3_tts(
    text: str = Form(...),
    language: str = Form("zh"),
    speed: float = Form(1.0),
    ref_text: str = Form(""),
    ref_audio: UploadFile = File(...),
    engine: str = Form("minimax_h3"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    job_dir = WORK_DIR / f"tts_{req_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    ref_path = job_dir / "ref.wav"
    raw_ref = job_dir / f"raw_{ref_audio.filename}"
    raw_ref.write_bytes(await ref_audio.read())
    trim_sec = "4.5" if engine == "hybrid" else "6"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(raw_ref),
        "-af", "silenceremove=start_periods=1:start_silence=0.05:start_threshold=-40dB,areverse,silenceremove=start_periods=1:start_silence=0.05:start_threshold=-40dB,areverse",
        "-t", trim_sec, "-ar", "24000", "-ac", "1", str(ref_path)
    ], check=True)

    clean_text = re.sub(r"\s+", " ", text.replace("\n", "，")).strip()
    if clean_text and clean_text[-1] not in "。！？.!?,，":
        clean_text += "。"

    if engine == "hybrid":
        H3_JOBS[req_id] = {"status": "running"}
        threading.Thread(
            target=_run_f5_tts_job,
            args=(req_id, job_dir, ref_path, clean_text, speed, ref_text),
            daemon=True,
        ).start()
        return {"job_id": req_id, "status": "running"}

    lang_tag = "English" if language == "en" else "Taiwanese Mandarin"
    tts_prompt = (
        "subject_definitions:\n"
        "<Subject 1> is an adult speaker with clear, natural articulation.\n"
        "<Audio 1> is the voice-timbre reference for <Subject 1> (S1).\n\n"
        "summary:\n"
        "[reference generation + audio reference] <Subject 1> delivers a clear spoken line.\n\n"
        "retention_analysis:\n"
        "<Audio 1>: reference - its vocal timbre guides the dialogue delivery of <Subject 1>.\n\n"
        "detailed_description:\n"
        f"[Shot 1] <Subject 1> (S1) speaks clearly and steadily, <d>[{lang_tag}] {clean_text}</d>"
    )

    manifest_path = job_dir / "manifest.json"
    manifest = {
        "jobs": [{
            "id": req_id,
            "type": "tts",
            "prompt": tts_prompt,
            "voices": ["ref.wav"],
            "lead_in": True,
        }]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    proc = subprocess.run(
        ["python", "/content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py", "--comfy", COMFY_URL, "--comfy-dir", "/content/ComfyUI", "--models-dir", "/content/ComfyUI/models", "run", str(manifest_path), "--confirmed"],
        capture_output=True,
        text=True,
    )
    wavs = list((job_dir / "out").glob("*.wav")) if (job_dir / "out").exists() else []
    if proc.returncode != 0 or not wavs:
        raise HTTPException(status_code=500, detail=f"MiniMax H3 TTS failed: {(proc.stdout + proc.stderr)[-600:]}")
    return FileResponse(str(wavs[0]), media_type="audio/wav", filename="h3_tts.wav")

def _run_h3_video_job(job_id: str, job_dir: Path, manifest_path: Path):
    try:
        proc = subprocess.run(
            ["python", "/content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py", "--comfy", COMFY_URL, "--comfy-dir", "/content/ComfyUI", "--models-dir", "/content/ComfyUI/models", "run", str(manifest_path), "--confirmed"],
            capture_output=True,
            text=True,
        )
        mp4s = list((job_dir / "out").glob("*.mp4")) if (job_dir / "out").exists() else []
        if proc.returncode != 0 or not mp4s:
            H3_JOBS[job_id] = {"status": "failed", "error": (proc.stdout + proc.stderr)[-600:]}
        else:
            H3_JOBS[job_id] = {"status": "done", "file": str(mp4s[0])}
    except Exception as exc:
        H3_JOBS[job_id] = {"status": "failed", "error": str(exc)}

@app.post("/api/lipsync/start")
async def h3_video_start(
    avatar: UploadFile = File(...),
    audio: UploadFile = File(...),
    engine: str = Form("minimax_h3"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    job_dir = WORK_DIR / f"vid_{req_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    ext = Path(avatar.filename or "face.png").suffix.lower() or ".png"
    face_name = f"face{ext}"
    (job_dir / face_name).write_bytes(await avatar.read())
    audio_file = job_dir / "audio.wav"
    audio_file.write_bytes(await audio.read())

    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(audio_file)],
        capture_output=True, text=True,
    )
    try:
        audio_dur = float(probe.stdout.strip())
    except Exception:
        audio_dur = 5.0
    video_sec = max(2.0, min(15.0, round(audio_dur + 0.3, 1)))

    manifest_path = job_dir / "manifest.json"
    manifest = {
        "jobs": [{
            "id": req_id,
            "type": "video",
            "mode": "ref",
            "inputs": [face_name],
            "drive_audio": "audio.wav",
            "seconds": video_sec,
            "aspect": "auto",
            "resolution": "480P",
            "prompt": "subject_definitions:\n<Subject 1> is the exact person in @圖片1.\n\n<Subject 1> speaks directly to the camera in sync with @音訊1, with natural subtle head movement, blinking, and identical face, clothing, and background as @圖片1.",
        }]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    H3_JOBS[req_id] = {"status": "running"}
    threading.Thread(target=_run_h3_video_job, args=(req_id, job_dir, manifest_path), daemon=True).start()
    return {"job_id": req_id, "status": "running"}

@app.get("/api/lipsync/status/{job_id}")
def h3_video_status(job_id: str, authorization: str | None = Header(default=None)):
    verify_token(authorization)
    job = H3_JOBS.get(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job not found")
    return {"job_id": job_id, "status": job["status"], "error": job.get("error")}

@app.get("/api/lipsync/result/{job_id}")
def h3_video_result(job_id: str, authorization: str | None = Header(default=None)):
    verify_token(authorization)
    job = H3_JOBS.get(job_id)
    if not job or job.get("status") != "done":
        raise HTTPException(status_code=404, detail="Result not ready")
    return FileResponse(job["file"], media_type="video/mp4", filename="h3_result.mp4")

@app.post("/api/lipsync")
async def h3_video(
    avatar: UploadFile = File(...),
    audio: UploadFile = File(...),
    engine: str = Form("minimax_h3"),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    job_dir = WORK_DIR / f"vid_{req_id}"
    job_dir.mkdir(parents=True, exist_ok=True)
    ext = Path(avatar.filename or "face.png").suffix.lower() or ".png"
    face_name = f"face{ext}"
    (job_dir / face_name).write_bytes(await avatar.read())
    audio_file = job_dir / "audio.wav"
    audio_file.write_bytes(await audio.read())

    # 量測語音真實秒數，確保生成影片長度完整不被截斷為 2 秒
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(audio_file)],
        capture_output=True, text=True,
    )
    try:
        audio_dur = float(probe.stdout.strip())
    except Exception:
        audio_dur = 5.0
    video_sec = max(2.0, min(15.0, round(audio_dur + 0.3, 1)))

    manifest_path = job_dir / "manifest.json"
    manifest = {
        "jobs": [{
            "id": req_id,
            "type": "video",
            "mode": "ref",
            "inputs": [face_name],
            "drive_audio": "audio.wav",
            "seconds": video_sec,
            "aspect": "auto",
            "resolution": "480P",
            "prompt": "subject_definitions:\n<Subject 1> is the exact person in @圖片1.\n\n<Subject 1> speaks directly to the camera in sync with @音訊1, with natural subtle head movement, blinking, and identical face, clothing, and background as @圖片1.",
        }]
    }
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False), encoding="utf-8")
    proc = subprocess.run(
        ["python", "/content/h3-colab-skill/h3-multimodal-colab/scripts/h3local.py", "--comfy", COMFY_URL, "--comfy-dir", "/content/ComfyUI", "--models-dir", "/content/ComfyUI/models", "run", str(manifest_path), "--confirmed"],
        capture_output=True,
        text=True,
    )
    mp4s = list((job_dir / "out").glob("*.mp4")) if (job_dir / "out").exists() else []
    if proc.returncode != 0 or not mp4s:
        raise HTTPException(status_code=500, detail=f"MiniMax H3 Video failed: {(proc.stdout + proc.stderr)[-600:]}")
    return FileResponse(str(mp4s[0]), media_type="video/mp4", filename="h3_result.mp4")

@app.post("/api/youtube-audio")
async def youtube_audio(
    url: str = Form(...),
    start_sec: float = Form(0.0),
    duration_sec: float = Form(12.0),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    raw_tpl = str(WORK_DIR / f"{req_id}_yt.%(ext)s")
    out_wav = WORK_DIR / f"{req_id}_yt_trim.wav"

    dl_proc = subprocess.run(
        ["yt-dlp", "-f", "bestaudio/best", "--no-playlist", "-o", raw_tpl, url.strip()],
        capture_output=True,
        text=True,
    )
    downloaded = next(WORK_DIR.glob(f"{req_id}_yt.*"), None)
    if dl_proc.returncode != 0 or not downloaded:
        raise HTTPException(status_code=400, detail=f"yt-dlp failed: {dl_proc.stderr[-300:]}")

    ff_proc = subprocess.run(
        [
            "ffmpeg", "-y",
            "-ss", str(max(0.0, start_sec)),
            "-t", str(min(45.0, max(3.0, duration_sec))),
            "-i", str(downloaded),
            "-ar", "24000", "-ac", "1",
            str(out_wav),
        ],
        capture_output=True,
        text=True,
    )
    downloaded.unlink(missing_ok=True)
    if ff_proc.returncode != 0 or not out_wav.exists():
        raise HTTPException(status_code=500, detail=f"ffmpeg trim failed: {ff_proc.stderr[-300:]}")

    return FileResponse(str(out_wav), media_type="audio/wav", filename="url_voice.wav")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8899, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

if TAILSCALE_AUTHKEY.strip():
    import json as _json
    print("🚀 正在啟動 Tailscale Funnel（免費固定 .ts.net 網址）...")
    subprocess.run("curl -fsSL https://tailscale.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["tailscaled", "--tun=userspace-networking", "--socks5-server=localhost:1055"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)
    subprocess.run(["tailscale", "up", f"--authkey={TAILSCALE_AUTHKEY.strip()}", "--hostname=ai-avatar-h3"], check=True)
    subprocess.run(["tailscale", "funnel", "--bg", "8899"], check=True)
    st = _json.loads(subprocess.check_output(["tailscale", "status", "--json"], text=True))
    dns_name = st["Self"]["DNSName"].rstrip(".")
    print("\n" + "=" * 60)
    print("✅ MiniMax H3 Colab Worker 已上線（Tailscale 固定網址）：")
    print(f"https://{dns_name}")
    print("=" * 60 + "\n")
    while True:
        time.sleep(60)
elif NGROK_AUTHTOKEN.strip() and NGROK_DOMAIN.strip():
    subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    domain = NGROK_DOMAIN.strip().replace("https://", "").replace("http://", "").strip("/")
    tunnel = ngrok.connect(8899, domain=domain)
    print("\n" + "=" * 60)
    print("✅ MiniMax H3 Colab Worker 已上線（固定網址 ngrok）：")
    print(f"https://{domain}")
    print("=" * 60 + "\n")
    while True:
        time.sleep(60)
elif CLOUDFLARE_TUNNEL_TOKEN.strip():
    print("🚀 正在啟動 MiniMax H3 Cloudflare Named Tunnel（固定網址）...")
    subprocess.run(["cloudflared", "tunnel", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN.strip()], check=True)
else:
    print("🚀 正在啟動 MiniMax H3 Cloudflare Tunnel...")
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8899"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in tunnel.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\n" + "=" * 60)
            print("✅ MiniMax H3 Colab Worker 已上線！")
            print(match.group(0))
            print("=" * 60 + "\n")